<a href="https://colab.research.google.com/github/Par-star/Agentic_AI/blob/main/Agentic_loop_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibilitỹ
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

✅ GPU available: Tesla T4
   Memory: 15.6 GB

📦 Python 3.12.13
🔥 PyTorch 2.11.0+cu128
🎲 Random seed set to 42


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
user_id = spark.conf.get("spark.foundry.user", "unknown_user")

print(f"The current user ID is: {user_id}")


The current user ID is: unknown_user


In [ ]:
import pandas as pd

# 1. Check a specific key without a default value (will raise error if missing)
try:
    val = spark.conf.get("spark.foundry.user")
    print(f"Found key: {val}")
except Exception as e:
    print(f"Key not found or error: {e}")

# 2. List all current Spark configurations to see what is actually set
all_conf = spark.sparkContext.getConf().getAll()
conf_df = pd.DataFrame(all_conf, columns=['Configuration Key', 'Value'])

display(conf_df)

Key not found or error: [SQL_CONF_NOT_FOUND] The SQL config "spark.foundry.user" cannot be found. Please verify that the config exists. SQLSTATE: 42K0I


,Configuration Key,Value
0,spark.app.id,local-1781862201412
1,spark.rdd.compress,True
2,spark.hadoop.fs.s3a.vectored.read.min.seek.size,128K
3,spark.app.submitTime,1781862199028
4,spark.driver.port,41161
5,spark.executor.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+Igno...
6,spark.sql.artifact.isolation.enabled,false
7,spark.sql.warehouse.dir,file:/content/spark-warehouse
8,spark.master,local[*]
9,spark.app.startTime,1781862199763


In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1eeAEwx5BNVDUgOULg1nC-ckX2cNMmv8f", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))

Downloading...
From: https://drive.google.com/uc?id=1eeAEwx5BNVDUgOULg1nC-ckX2cNMmv8f
To: /content/narration.zip
100%|██████████| 6.09M/6.09M [00:00<00:00, 40.4MB/s]


Loaded 14 narration segments


In [ ]:
import random

# To keep a difference of 13 within a max range of 24,
# the smallest number (first_number) cannot exceed 11.
# (11 + 13 = 24)
first_number = random.randint(0, 11)

# Calculate the second number
second_number = first_number + 13

# Output the results
print(f"First Number (Smallest): {first_number}")
print(f"Second Number (Largest): {second_number}")
print(f"Difference: {second_number - first_number}")

First Number (Smallest): 10
Second Number (Largest): 23
Difference: 13


In [ ]:
# Setup
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

# Traditional approach
def handle_query_traditional(query):
    order_id = "ORD-1234"
    status = {"status": "shipped"}
    message = "What is my order status"
    return f"Message {message} : Order {order_id}: {status}"

# Agentic approach — model decides everything
# The following loop is conceptual and causes a NameError as 'call_model' is not defined in this cell.
# The actual agentic loop implementation is in later cells.
# while True:
#      response = call_model(messages)
#      if response.stop_reason == "end_turn":
#       break
#      if response.stop_reason == "tool_use":
#        execute_tool_and_append_result()

print("Core insight: the MODEL drives the loop, not the developer.")
print("The stop_reason field is the steering wheel.")
print()
print("Traditional:", handle_query_traditional("order status"))

Core insight: the MODEL drives the loop, not the developer.
The stop_reason field is the steering wheel.

Traditional: Message What is my order status : Order ORD-1234: {'status': 'shipped'}


In [ ]:
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
from typing import Any, Dict, List, Optional

def handle_query_traditional(query):
  order_id = "ORD-1234"
  status = {"status": "shipped"}

  message = "Your order status - "
  return f"{message}: Order {order_id}: {status}"

handle_query_traditional("Any updates on the order")


"Your order status - : Order ORD-1234: {'status': 'shipped'}"

In [ ]:
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = field(default_factory=lambda: f"toolu_{uuid.uuid4().hex[:12]}")
    name: str = ""
    input: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class MockResponse:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    model: str = "claude-sonnet-4-20250514"

ORDER_DB = {
    "ORD-1234": {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99},
    "ORD-5555": {"status": "processing", "tracking": None, "item": "USB-C Cable", "amount": 12.99},
    "ORD-9999": {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand", "amount": 45.00},
}

SHIPPING_DB = {
    "TRK-5678": {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"},
    "TRK-1111": {"carrier": "UPS", "eta": "2024-03-10", "last_location": "Delivered"},
}

def execute_tool(tool_name, tool_input):
    if tool_name == "lookup_order":
        oid = tool_input.get("order_id", "")
        return json.dumps(ORDER_DB.get(oid, {"error": f"Not found: {oid}"}))
    elif tool_name == "check_shipping":
        tid = tool_input.get("tracking_id", "")
        return json.dumps(SHIPPING_DB.get(tid, {"error": f"Not found: {tid}"}))
    elif tool_name == "process_refund":
        return json.dumps({"success": True, "refund_id": f"REF-{uuid.uuid4().hex[:6]}", "amount": tool_input.get("amount", 0)})
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

print("lookup_order:", execute_tool("lookup_order", {"order_id": "ORD-1234"}))
print("check_shipping:", execute_tool("check_shipping", {"tracking_id": "TRK-5678"}))

lookup_order: {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99}
check_shipping: {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"}


In [ ]:
class MockClaudeClient:
    def __init__(self):
        self.call_count = 0
        self.scenario = []

    def set_scenario(self, responses):
        self.scenario = responses
        self.call_count = 0

    def create_message(self, model, max_tokens, tools, messages):
        if self.call_count < len(self.scenario):
            r = self.scenario[self.call_count]
            self.call_count += 1
            return r
        return MockResponse(content=[TextBlock(text="Done.")], stop_reason="end_turn")


TOOLS = [
    {"name": "lookup_order", "description": "Look up order by ID",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
    {"name": "check_shipping", "description": "Check shipping by tracking ID",
     "input_schema": {"type": "object", "properties": {"tracking_id": {"type": "string"}}, "required": ["tracking_id"]}},
    {"name": "process_refund", "description": "Process a refund",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
]


client = MockClaudeClient()
print(f"Mock client ready with {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")

Mock client ready with 3 tools: ['lookup_order', 'check_shipping', 'process_refund']


In [ ]:
def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn - {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")

Agentic loop defined!


In [ ]:
#@title 🎧 Listen: Tracing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_tracing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Tracing a Multi-Turn Execution

In [ ]:
client = MockClaudeClient()
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="check_shipping", input={"tracking_id": "TRK-5678"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Your order ORD-1234 (Headphones, $79.99) shipped via FedEx. In Memphis, TN. Arriving March 15. Tracking: TRK-5678.")], stop_reason="end_turn"),
])
result = 0

run_agentic_loop(client, TOOLS, "What is the status of order ORD-1234?")

User: What is the status of order ORD-1234?

--- Turn - 1 | stop_reason: tool_use ---
Agent calls: lookup_order({"order_id": "ORD-1234"})
Tool result: {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99}

--- Turn - 2 | stop_reason: tool_use ---
Agent calls: check_shipping({"tracking_id": "TRK-5678"})
Tool result: {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"}

--- Turn - 3 | stop_reason: end_turn ---
Agent (final): Your order ORD-1234 (Headphones, $79.99) shipped via FedEx. In Memphis, TN. Arriving March 15. Tracking: TRK-5678.


'Your order ORD-1234 (Headphones, $79.99) shipped via FedEx. In Memphis, TN. Arriving March 15. Tracking: TRK-5678.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

**Turn 1:** Model called `lookup_order`. **Turn 2:** Called `check_shipping`. **Turn 3:** Had all info, responded with `end_turn`. The model made every decision. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Anti Patterns
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_anti_patterns.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Anti-Pattern Detection Lab

### Anti-Pattern 1: Natural Language Parsing

In [ ]:
def broken_loop_v1(client, tools, user_message, max_turns=10):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        for block in response.content:
            if hasattr(block, "text") and "done" in block.text.lower():
                return f"[EARLY EXIT] {block.text}"
        if response.stop_reason == "tool_use":
            for block in response.content:
                if hasattr(block, "name"):
                    result = execute_tool(block.name, block.input)
                    messages.append({"role": "assistant", "content": response.content})
                    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": block.id, "content": result}]})
                    break
    return "Max turns"

client.set_scenario([
    MockResponse(content=[TextBlock(text="Done checking DB, now shipping..."), ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Shipped via FedEx.")], stop_reason="end_turn"),
])
print("Anti-Pattern 1: NL parsing")
print(f"Result: {broken_loop_v1(client, TOOLS, 'Check ORD-1234')}")
print("Bug: 'done checking' triggered exit mid-task!")

Anti-Pattern 1: NL parsing
Result: [EARLY EXIT] Done checking DB, now shipping...
Bug: 'done checking' triggered exit mid-task!


In [ ]:
#@title 🎧 Listen: Anti Pattern 2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_anti_pattern_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Anti-Pattern 2: Iteration Caps

In [ ]:
def broken_loop_v2(client, tools, msg):
    messages = [{"role": "user", "content": msg}]
    for i in range(3):
        r = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        if r.stop_reason == "tool_use":
            for b in r.content:
                if hasattr(b, "name"):
                    res = execute_tool(b.name, b.input)
                    messages.append({"role": "assistant", "content": r.content})
                    messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": b.id, "content": res}]})
                    break
    return "Completed 3 iterations"

# Re-initializing client here to ensure it's defined
client = MockClaudeClient()
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-5555"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="ORD-5555 processing.")], stop_reason="end_turn"),
    MockResponse(content=[TextBlock(text="Already answered.")], stop_reason="end_turn"),
])
print("Anti-Pattern 2: Iteration cap")
print(f"Result: {broken_loop_v2(client, TOOLS, 'Status ORD-5555?')}")
print("Bug: Model done in turn 2, loop forced turn 3!")

Anti-Pattern 2: Iteration cap
Result: Completed 3 iterations
Bug: Model done in turn 2, loop forced turn 3!


### Comparison

In [ ]:
print(f"{'Scenario':<25} {'Correct':<10} {'Cap=3':<10} {'Issue'}")
print("=" * 58)
for n, t in [("2-turn", 2), ("1-turn", 1), ("3-turn", 3), ("5-turn", 5)]:
    w = max(0, 3 - t)
    m = max(0, t - 3)
    p = f"{w} wasted" if w else (f"{m} MISSED" if m else "OK")
    print(f"  {n:<23} {t:<10} {3:<10} {p}")

Scenario                  Correct    Cap=3      Issue
  2-turn                  2          3          1 wasted
  1-turn                  1          3          2 wasted
  3-turn                  3          3          OK
  5-turn                  5          3          2 MISSED


In [ ]:
#@title 🎧 Listen: Todo Exercises

from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_todo_exercises.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Your Turn — TODO Exercises

### Exercise 1: Refund Agent

In [ ]:
# ============ TODO ============
# 3-turn scenario: lookup ORD-9999, process_refund($45), confirm
client = MockClaudeClient()
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-9999"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-9999", "amount": 45})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Refund REF-9999 processed for $45.")], stop_reason="end_turn")
])
result = run_agentic_loop(client, TOOLS, "Refund ORD-9999")
print(result)
# ==============================

User: Refund ORD-9999

--- Turn - 1 | stop_reason: tool_use ---
Agent calls: lookup_order({"order_id": "ORD-9999"})
Tool result: {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand", "amount": 45.0}

--- Turn - 2 | stop_reason: tool_use ---
Agent calls: process_refund({"order_id": "ORD-9999", "amount": 45})
Tool result: {"success": true, "refund_id": "REF-3fb98c", "amount": 45}

--- Turn - 3 | stop_reason: end_turn ---
Agent (final): Refund REF-9999 processed for $45.
Refund REF-9999 processed for $45.


In [ ]:
#@title 🎧 Listen: Todo Fix Loop
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo_fix_loop.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:

# This is formatted as code
### Exercise 2: Fix the Broken Loop

def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        print("entered")
        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")

Agentic loop defined!




```
# This is formatted as code
### Exercise 2: Fix the Broken Loop

def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")


```



In [ ]:

# This is formatted as code
### Exercise 2: Fix the Broken Loop

def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")






# ==================================================================================================================================================



def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")

Agentic loop defined!
Agentic loop defined!


In [ ]:
import json
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

@dataclass
class ToolUseBlock:
    type: str = "tool_use"
    id: str = field(default_factory=lambda: f"toolu_{uuid.uuid4().hex[:12]}")
    name: str = ""
    input: Dict[str, Any] = field(default_factory=dict)

@dataclass
class TextBlock:
    type: str = "text"
    text: str = ""

@dataclass
class MockResponse:
    content: list = field(default_factory=list)
    stop_reason: str = "end_turn"
    model: str = "claude-sonnet-4-20250514"

ORDER_DB = {
    "ORD-1234": {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99},
    "ORD-5555": {"status": "processing", "tracking": None, "item": "USB-C Cable", "amount": 12.99},
    "ORD-9999": {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand", "amount": 45.00},
}

SHIPPING_DB = {
    "TRK-5678": {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"},
    "TRK-1111": {"carrier": "UPS", "eta": "2024-03-10", "last_location": "Delivered"},
}

def execute_tool(tool_name, tool_input):
    if tool_name == "lookup_order":
        oid = tool_input.get("order_id", "")
        return json.dumps(ORDER_DB.get(oid, {"error": f"Not found: {oid}"}))
    elif tool_name == "check_shipping":
        tid = tool_input.get("tracking_id", "")
        return json.dumps(SHIPPING_DB.get(tid, {"error": f"Not found: {tid}"}))
    elif tool_name == "process_refund":
        return json.dumps({"success": True, "refund_id": f"REF-{uuid.uuid4().hex[:6]}", "amount": tool_input.get("amount", 0)})
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

class MockClaudeClient:
    def __init__(self):
        self.call_count = 0
        self.scenario = []

    def set_scenario(self, responses):
        self.scenario = responses
        self.call_count = 0

    def create_message(self, model, max_tokens, tools, messages):
        if self.call_count < len(self.scenario):
            r = self.scenario[self.call_count]
            self.call_count += 1
            return r
        return MockResponse(content=[TextBlock(text="Done.")], stop_reason="end_turn")


TOOLS = [
    {"name": "lookup_order", "description": "Look up order by ID",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}},
    {"name": "check_shipping", "description": "Check shipping by tracking ID",
     "input_schema": {"type": "object", "properties": {"tracking_id": {"type": "string"}}, "required": ["tracking_id"]}},
    {"name": "process_refund", "description": "Process a refund",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": "number"}}, "required": ["order_id", "amount"]}},
]


client = MockClaudeClient()


# This is formatted as code
# This comment just says this section is being shown as code

### Exercise 2: Fix the Broken Loop
# This is just a title/comment for the exercise
# It is not Python logic you need for the loop itself


def run_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    # Define a function named run_agentic_loop
    # client = model/client object
    # tools = list of tools the model can use
    # user_message = first message from user
    # max_turns=10 = safety limit so loop does not run forever
    # verbose=True = print step-by-step logs

    messages = [{"role": "user", "content": user_message}]
    # Start conversation history with one user message

    turn = 0
    # Start turn counter at 0

    if verbose:
        # Check if logging/printing is enabled

        print(f"{'='*60}")
        # Print 60 equal signs to make output easier to read

        print(f"User: {user_message}")
        # Print the user's message

        print(f"{'='*60}")
        # Print another separator line

    while turn < max_turns:
        # Keep looping until turn count reaches max_turns
        print(turn)

        turn += 1
        # Increase turn count by 1 each time loop runs

        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)
        # Send the current conversation to the model
        # model=... chooses the model name
        # max_tokens=1024 sets output size limit
        # tools=tools gives tool access to the model
        # messages=messages sends the chat history

        if verbose:
            # If logging is enabled

            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")
            # Print current turn number and why the model stopped
            # stop_reason usually tells whether model is done or wants a tool

        if response.stop_reason == "end_turn":
            # If model says it is finished answering

            final_text = ""
            # Create empty string to store final answer text

            for block in response.content:
                # Go through each content block in the response

                if hasattr(block, "text"):
                    # Check whether this block has a text field

                    final_text = block.text
                    # Save the text as final_text
                    # If there are multiple text blocks, this keeps the last one found

            if verbose:
                # If logging is enabled

                print(f"Agent (final): {final_text}")
                # Print the final answer from the agent

            return final_text
            # Return final answer and stop function

        elif response.stop_reason == "tool_use":
            # If model says it wants to use a tool

            tool_block = None
            # Create variable to store the tool call block

            for block in response.content:
                # Go through each block in model response

                if hasattr(block, "name"):
                    # Check whether this block has a name field
                    # Tool blocks usually have a tool name

                    tool_block = block
                    # Save this block as the tool block

                    break
                    # Stop after finding the first tool block

            if tool_block is None:
                # If no tool block was found even though stop_reason said tool_use

                break
                # Exit loop because something is wrong or incomplete

            if verbose:
                # If logging is enabled

                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")
                # Print which tool the model is calling
                # json.dumps makes the input dictionary print nicely as JSON-like text

            result = execute_tool(tool_block.name, tool_block.input)
            # Actually run the tool using tool name and tool input
            # Save returned result in result variable

            if verbose:
                # If logging is enabled

                print(f"Tool result: {result}")
                # Print the result returned by the tool



            messages.append({"role": "assistant", "content": response.content})
            # Add the assistant's tool-call response into the conversation history
            # This preserves what the model asked for

            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})
            # Add the tool result back into conversation history
            # It is added in special tool_result format
            # tool_use_id links this result to the exact tool call that requested it

    return "Max turns reached (safety net)"
    # If loop never got end_turn before hitting limit, return this fallback message


print("Agentic loop defined!")
# Print confirmation that function is now created
client = MockClaudeClient() # Re-initialize client for this cell
result = run_agentic_loop(client=client, tools=TOOLS, user_message="Tell me a joke.", verbose=True)
print(f"The final result was: {result}")

Agentic loop defined!
User: Tell me a joke.
0

--- Turn 1 | stop_reason: end_turn ---
Agent (final): Done.
The final result was: Done.


In [ ]:
### Exercise 2: Fix the Broken Loop

def fixed_agentic_loop(client, tools, user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {user_message}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"

print("Agentic loop defined!")


Agentic loop defined!


In [ ]:

# ============ TODO ============
def fixed_loop(client, tools, msg, max_turns=10):
    messages = [{"role": "user", "content": msg}]

    turn = 0
    verbose = True
    if verbose:
        print(f"{'='*60}")
        print(f"User: {msg}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")
            return final_text

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)"
# ==============================

In [ ]:
#@title 🎧 Listen: Todo Logger
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_todo_logger.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Exercise 3: Conversation Logger

In [ ]:
def _to_json_serializable(obj):
    if isinstance(obj, ToolUseBlock):
        return {"type": obj.type, "id": obj.id, "name": obj.name, "input": obj.input}
    if isinstance(obj, TextBlock):
        return {"type": obj.type, "text": obj.text}
    return obj

def logging_loop(client, tools, msg, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": msg}]
    turn = 0

    if verbose:
        print(f"{'='*60}")
        print(f"User: {msg}")
        print(f"{'='*60}")

    while turn < max_turns:
        turn += 1
        response = client.create_message(model="claude-sonnet-4-20250514", max_tokens=1024, tools=tools, messages=messages)

        if verbose:
            print(f"\n--- Turn {turn} | stop_reason: {response.stop_reason} ---")

        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
            if verbose:
                print(f"Agent (final): {final_text}")

            # Convert content to serializable format before appending
            serializable_content = [_to_json_serializable(block) for block in response.content]
            messages.append({"role": "assistant", "content": serializable_content})

            return final_text, messages

        elif response.stop_reason == "tool_use":
            tool_block = None
            for block in response.content:
                if hasattr(block, "name"):
                    tool_block = block
                    break
            if tool_block is None:
                break

            if verbose:
                print(f"Agent calls: {tool_block.name}({json.dumps(tool_block.input)})")

            result = execute_tool(tool_block.name, tool_block.input)
            if verbose:
                print(f"Tool result: {result}")

            # Convert response.content to serializable format before appending
            serializable_content = [_to_json_serializable(block) for block in response.content]
            messages.append({"role": "assistant", "content": serializable_content})
            messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}]})

    return "Max turns reached (safety net)", messages

print("Conversation logging loop defined!")

Conversation logging loop defined!


In [ ]:
client = MockClaudeClient()
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-9999"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="process_refund", input={"order_id": "ORD-9999", "amount": 45})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Refund for ORD-9999 processed for $45.")], stop_reason="end_turn")
])

final_response, conversation_history = logging_loop(client, TOOLS, "I need to refund order ORD-9999.", verbose=True)
print(f"\nFinal Agent Response: {final_response}")
print(f"\nFull Conversation History: {json.dumps(conversation_history, indent=2)}")

User: I need to refund order ORD-9999.

--- Turn 1 | stop_reason: tool_use ---
Agent calls: lookup_order({"order_id": "ORD-9999"})
Tool result: {"status": "delivered", "tracking": "TRK-1111", "item": "Laptop Stand", "amount": 45.0}

--- Turn 2 | stop_reason: tool_use ---
Agent calls: process_refund({"order_id": "ORD-9999", "amount": 45})
Tool result: {"success": true, "refund_id": "REF-7dda6e", "amount": 45}

--- Turn 3 | stop_reason: end_turn ---
Agent (final): Refund for ORD-9999 processed for $45.

Final Agent Response: Refund for ORD-9999 processed for $45.

Full Conversation History: [
  {
    "role": "user",
    "content": "I need to refund order ORD-9999."
  },
  {
    "role": "assistant",
    "content": [
      {
        "type": "tool_use",
        "id": "toolu_c3f04aaaf39a",
        "name": "lookup_order",
        "input": {
          "order_id": "ORD-9999"
        }
      }
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "tool_result",
        "too

In [ ]:
#@title 🎧 Listen: Putting Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_putting_together.mp3"
if _os.path.exists(_f):
  \

    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Run the first cell to download narration audio.


## 8. Putting It All Together

In [ ]:
client = MockClaudeClient()
client.set_scenario([
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-1234"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="check_shipping", input={"tracking_id": "TRK-5678"})], stop_reason="tool_use"),
    MockResponse(content=[ToolUseBlock(name="lookup_order", input={"order_id": "ORD-5555"})], stop_reason="tool_use"),
    MockResponse(content=[TextBlock(text="Both orders:\n1. ORD-1234: Shipped, FedEx, March 15\n2. ORD-5555: Processing, 1-2 days")], stop_reason="end_turn"),
])
run_agentic_loop(client, TOOLS, "Check orders ORD-1234 and ORD-5555")

User: Check orders ORD-1234 and ORD-5555
0

--- Turn 1 | stop_reason: tool_use ---
Agent calls: lookup_order({"order_id": "ORD-1234"})
Tool result: {"status": "shipped", "tracking": "TRK-5678", "item": "Wireless Headphones", "amount": 79.99}
1

--- Turn 2 | stop_reason: tool_use ---
Agent calls: check_shipping({"tracking_id": "TRK-5678"})
Tool result: {"carrier": "FedEx", "eta": "2024-03-15", "last_location": "Memphis, TN"}
2

--- Turn 3 | stop_reason: tool_use ---
Agent calls: lookup_order({"order_id": "ORD-5555"})
Tool result: {"status": "processing", "tracking": null, "item": "USB-C Cable", "amount": 12.99}
3

--- Turn 4 | stop_reason: end_turn ---
Agent (final): Both orders:
1. ORD-1234: Shipped, FedEx, March 15
2. ORD-5555: Processing, 1-2 days


'Both orders:\n1. ORD-1234: Shipped, FedEx, March 15\n2. ORD-5555: Processing, 1-2 days'

In [ ]:
#@title 🎧 Listen: Exam Practice
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_exam_practice.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Run the first cell to download narration audio.


## 9. Exam Practice Questions

In [ ]:
for i, (q, a, w) in enumerate([
    ("stop_reason='tool_use' but text says 'we are done.' Action?", "Execute tool, continue loop", "stop_reason is authoritative."),
    ("for i in range(5): call_model() — what's wrong?", "Cap is primary control, not stop_reason", "Model drives via end_turn."),
    ("After tool exec, what's appended?", "Assistant response + user tool_result", "Both required by API."),
    ("Two stop_reason values for agentic loop?", "tool_use and end_turn", "tool_use=need tool, end_turn=done."),
], 1):
    print(f"Q{i}: {q}\n   {a} — {w}\n")

Q1: stop_reason='tool_use' but text says 'we are done.' Action?
   Execute tool, continue loop — stop_reason is authoritative.

Q2: for i in range(5): call_model() — what's wrong?
   Cap is primary control, not stop_reason — Model drives via end_turn.

Q3: After tool exec, what's appended?
   Assistant response + user tool_result — Both required by API.

Q4: Two stop_reason values for agentic loop?
   tool_use and end_turn — tool_use=need tool, end_turn=done.



In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Run the first cell to download narration audio.


# New section

## 10. Reflection

**Key exam concepts (Task Statement 1.1):**
- Agentic loop checks `stop_reason` after every API call
- `tool_use` → execute, append, loop. `end_turn` → done, exit.
- Never parse NL for termination. Never use iteration caps as primary control.
- Append BOTH assistant response AND tool result.

In [ ]:
print("Notebook complete! Next: Multi-Agent Coordinator Patterns (NB 2)")

In [ ]:

import shutil
from google.colab import files

# 1. Compress the folder into a zip file
# This creates 'narration_backup.zip' from the '/content/narration' directory
shutil.make_archive('narration_backup', 'zip', '/content/narration')

# 2. Download the zip file to your local machine
files.download('narration_backup.zip')